# Practica 3 — Procesamiento Digital de Imagenes

**Alumno:** Fernando Leon Franco
**Fecha de entrega:** _por definir_

---

## Resumen

_TODO: describir a grandes rasgos el objetivo y las tecnicas utilizadas en el desarrollo de la practica._

Esta practica aplica filtros basados en mascara de convolucion para resolver problemas de ruido y mejora de imagenes. Las tecnicas a usar son: filtrado pasa-bajas iterado, filtros para eliminacion de ruido (mediana, gaussiano, etc.), filtro de mediana sobre ruido uniforme, High-Boost combinado con ajustes de contraste y ecualizacion, deteccion de bordes, y reduccion de ruido por resta y promedio de imagenes consecutivas.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path
import matplotlib.pyplot as plt

BASE_DIR = Path().resolve().parents[1]
sys.path.insert(0, str(BASE_DIR))

from image_processing import DerivativeVisionNode

DATA_DIR = BASE_DIR / "image_processing" / "practica_3" / "data"

files = {
    # E1, E3 — imagen base para pruebas
    "golf":         "golf.bmp",
    # E2 — eliminacion de ruido
    "torre_verona": "torre_verona.bmp",
    "florencia":    "florencia.bmp",
    "florencia1":   "florencia1.bmp",
    # E4 — radiografia para High-Boost
    "torax":        "toraxP.bmp",
    # E5 — deteccion de bordes
    "waldo":        "waldo.bmp",
    # E6, E7 — fibra optica (la lista de 11 se genera abajo)
    # naming sugerido: fibra_01.bmp, fibra_02.bmp, ..., fibra_11.bmp
}

def load_image(nombre: str) -> DerivativeVisionNode:
    return DerivativeVisionNode.desde_archivo(DATA_DIR / files[nombre])

# Helper para cargar las 11 fibras del E6/E7 (ajustar naming si difiere)
def load_fibras(n: int = 11) -> list[DerivativeVisionNode]:
    return [
        DerivativeVisionNode.desde_archivo(DATA_DIR / f"fibra_{i:02d}.bmp")
        for i in range(1, n + 1)
    ]

## 1️⃣ — Ejercicio 1

Aplique el filtrado pasa-bajas varias veces sobre la misma imagen. Explique el resultado limite de aplicar este filtro un numero infinito de veces.

In [ ]:
imagen = load_image("golf")
imagen.mostrar(block=False)
imagen.histograma(block=False)

# TODO: aplicar imagen.gaussiano(size=5, sigma=1.0) varias veces sobre la misma imagen.
#       Equivalentes: imagen.suavizar(size) o imagen.piramidal(size).
# TODO: mostrar imagen + histograma en algunas iteraciones (p.ej. 1, 5, 20, 100).
# TODO: observar a qué tiende la imagen en el limite (pista: media local que se propaga).

plt.show()

### Explicaciones del Ejercicio 1

**Descripcion:** _TODO_

**¿Cual es el resultado limite de aplicar el filtro pasa-bajas un numero infinito de veces?**

_TODO_

## 2️⃣ — Ejercicio 2

Elimine el ruido presente en las fotografias **torre_verona.bmp**, **florencia.bmp** y **florencia1.bmp**. Describa de forma cualitativa y cuantitativa el tipo de ruido presente en la imagen. Justifique el filtro utilizado, asi como los valores de tamaño de ventana, umbral, etc.

In [ ]:
imagen_torre     = load_image("torre_verona")
imagen_florencia = load_image("florencia")
imagen_florencia1 = load_image("florencia1")

for img in (imagen_torre, imagen_florencia, imagen_florencia1):
    img.mostrar(block=False)
    img.histograma(block=False)

# TODO: para cada imagen, identificar el tipo de ruido y elegir filtro:
#   - sal y pimienta (impulsivo) → img.mediana(size) o img.mediana_cruz(size)
#   - gaussiano / uniforme        → img.gaussiano(size, sigma) o img.filtro_sigma(size, sigma)
# TODO: usar img.senal_por_canal() (slider de fila) para ver la pinta del ruido.
# TODO: opcional — img.filtro_umbral(filtrada, umbral=0.15) para preservar detalle
#       en zonas sin ruido (mezcla original con la filtrada solo donde la diferencia es grande).
# TODO: justificar la elección del filtro y los parámetros (tamaño de ventana, umbral, σ).

plt.show()

### Explicaciones del Ejercicio 2

**Descripcion:** _TODO_

**Tipo de ruido (cualitativo y cuantitativo) presente en cada imagen:**

- torre_verona.bmp: _TODO_
- florencia.bmp: _TODO_
- florencia1.bmp: _TODO_

**Justificacion del filtro y de los parametros (ventana, umbral, etc.):**

_TODO_

## 3️⃣ — Ejercicio 3

Utilizando un generador de ruido uniforme, pruebe el filtro de mediana con distintos valores de radio y porcentajes de ruido, en tonos de grises y en RGB. En base a sus pruebas ¿es posible establecer una relacion entre porcentaje de ruido y radio de mascara de filtrado? Explique.

In [ ]:
imagen_color  = load_image("golf")
imagen_grises = imagen_color.escala_grises()
imagen_color.mostrar(block=False)
imagen_grises.mostrar(block=False)

# TODO: img_ruidosa = img.ruido_uniforme(amplitud=..., seed=42)
#       Probar varias amplitudes (p.ej. 0.05, 0.10, 0.20).
# TODO: img_ruidosa.mediana(size=...) con varios sizes (3, 5, 7).
# TODO: repetir el barrido en grises (imagen_grises) y en color (imagen_color).
# TODO: comparar con histograma — la mediana cuadrada y la cruz tienen sesgos distintos.
# TODO: comentar la relación entre amplitud_ruido y size del filtro
#       (a más ruido → ¿hace falta más ventana? ¿se pierde detalle?).

plt.show()

### Explicaciones del Ejercicio 3

**Descripcion:** _TODO_

**¿Es posible establecer una relacion entre porcentaje de ruido y radio de mascara de filtrado?**

_TODO_

**Diferencias entre procesamiento en grises y en RGB:**

_TODO_

## 4️⃣ — Ejercicio 4

Utilizando la tecnica **High-Boost**, en conjunto con los ajustes de contraste y ecualizacion probados anteriormente, busque un mejor resultado para el tratamiento de la radiografia. Pruebe ajustar primero el contraste seguido del filtrado, asi como filtrado primero y terminar con ajuste de contraste. ¿Hay alguna diferencia en el orden?

In [ ]:
imagen_torax = load_image("torax").escala_grises()
imagen_torax.mostrar(block=False)
imagen_torax.histograma(block=False)

# High-Boost en radiografía. Receta: img - k * laplaciano(crudo=True), terminar con .clip().
# (esto es lo que ya usaste en proyecto_costal.py — aquí sin AGNF de por medio).
#
# TODO (orden A: contraste -> high-boost):
#   ajustada  = imagen_torax.ecualizar()    # o .transformacion_gamma(...) o .transformacion_lineal(...)
#   lap       = ajustada.laplaciano(extendido=True, crudo=True)
#   resultado = (ajustada - lap * k).clip() # k típico: 0.5 a 2.0
#
# TODO (orden B: high-boost -> contraste):
#   lap       = imagen_torax.laplaciano(extendido=True, crudo=True)
#   boosted   = (imagen_torax - lap * k).clip()
#   resultado = boosted.ecualizar()         # mismo ajuste que en A
#
# TODO: comparar A vs B (imagen + histograma + zoom a la zona pulmón/cadera).
# TODO: explicar si el orden cambia el resultado (sí cambia: la ecualización post-boost
#       redistribuye los bordes amplificados; pre-boost amplifica bordes ya estirados).

plt.show()

### Explicaciones del Ejercicio 4

**Descripcion:** _TODO_

**¿Hay alguna diferencia en el orden (contraste→filtrado vs filtrado→contraste)?**

_TODO_

## 5️⃣ — Ejercicio 5

Elija una fotografia para aplicar la tecnica de deteccion de bordes. El resultado del procesamiento debe parecer el trazado en blanco y negro del perfil de los objetos.

In [ ]:
imagen = load_image("waldo").escala_grises()
imagen.mostrar(block=False)
imagen.histograma(block=False)

# Detección de bordes -> trazado en blanco y negro del perfil de los objetos.
#
# TODO: aplicar laplaciano:  bordes = imagen.laplaciano(extendido=True)
#       extendido=True usa kernel de 8 vecinos (incluye diagonales) → más isotrópico.
# TODO: opcionalmente realzar el resultado:
#       bordes.estirar_contraste()   # mapea min-max a [0,1]
#       bordes.binarizar(0.1)         # umbral global → solo blanco/negro
# TODO: comparar bordes sobre original vs sobre imagen.gaussiano(5, 1) primero
#       (suavizar antes del laplaciano = LoG, reduce ruido en los bordes).

plt.show()

### Explicaciones del Ejercicio 5

**Descripcion:** _TODO_

**Operador elegido y por que:**

_TODO_

## 6️⃣ — Ejercicio 6

En la carpeta **FIBRA** hay multiples imagenes de una fibra optica dopada con Erbio cuyo nucleo brilla en color verde cuando se bombea con un diodo laser de 1500 nm. Estas imagenes fueron tomadas en condiciones de baja iluminacion y con una camara que era particularmente ruidosa. Realice la **resta entre dos imagenes consecutivas** para evaluar el ruido. Puede usar el histograma para revisar si el ruido tiene una distribucion normal.

In [ ]:
fibras = load_fibras(11)
fibra_a = fibras[0].escala_grises()
fibra_b = fibras[1].escala_grises()
fibra_a.mostrar(block=False)
fibra_b.mostrar(block=False)

# Resta entre dos imágenes consecutivas para evaluar el ruido.
# OJO: como la señal real es la misma en ambas, el residuo queda ~puro ruido.
#
# TODO: residuo = (fibra_a - fibra_b).clip()
# TODO: residuo.histograma(block=False)
#       Si el ruido es normal se ve como una campana — pero la resta tiene valores
#       negativos que el .clip aplasta a 0. Para verlo bien, sumar offset visual:
#       diff_centrada = (fibra_a - fibra_b + 0.5).clip()
# TODO: comentar si la pinta del histograma sugiere distribución normal.

plt.show()

### Explicaciones del Ejercicio 6

**Descripcion:** _TODO_

**¿El ruido tiene distribucion normal? (justificar con histograma):**

_TODO_

## 7️⃣ — Ejercicio 7

Otra forma de reducir el ruido sin atenuar las altas frecuencias es **promediar imagenes consecutivas**. Esta tecnica se usa frecuentemente en Astronomia. Promedie 11 imagenes consecutivas de la fibra y compare el resultado con la imagen original. ¿Puede apreciar la reduccion del ruido?

In [ ]:
fibras = [f.escala_grises() for f in load_fibras(11)]
fibras[0].mostrar(block=False)
fibras[0].histograma(block=False)

# Promediar 11 imágenes consecutivas → reduce ruido sin atenuar altas frecuencias.
#
# TODO: promedio = fibras[0].promediar(fibras[1:])
# TODO: comparar fibras[0].mostrar() vs promedio.mostrar() y sus histogramas.
# TODO: explicar por qué reduce ruido (σ_promedio = σ / √N, con N=11 → factor √11 ≈ 3.3).

plt.show()

### Explicaciones del Ejercicio 7

**Descripcion:** _TODO_

**¿Se aprecia la reduccion del ruido? Justificar:**

_TODO_

## Conclusiones

_TODO: cuando son utiles y cuando no funcionan bien las tecnicas usadas — pasa-bajas, mediana, gaussiano, High-Boost, deteccion de bordes, resta y promedio de imagenes._